In [0]:
-- ============================================
-- BRIGHTTV DATA EXPLORATION & PREPARATION
-- ============================================

-- Initial data exploration: Inspect both core tables
-- Purpose: Understand structure, column types, and sample records
SELECT * 
FROM workspace.default.bright_tv_user_profiles;

SELECT * 
FROM workspace.default.bright_tv_viewership;


-- --------------------------------------------
-- DATE RANGE ANALYSIS
-- --------------------------------------------
-- Determine the coverage period of the dataset
-- Helps validate whether the data aligns with the intended analysis window
SELECT 
    MIN(RecordDate2) AS start_date,
    MAX(RecordDate2) AS end_date
FROM workspace.default.bright_tv_viewership;


-- --------------------------------------------
-- VOLUME ANALYSIS
-- --------------------------------------------
-- Total number of viewership records (proxy for total views)
SELECT 
    COUNT(`Duration 2`) AS total_views
FROM workspace.default.bright_tv_viewership;

-- Count total records in each table
-- User profiles ≈ 5,375 | Viewership ≈ 10,000
SELECT COUNT(*) AS total_users 
FROM workspace.default.bright_tv_user_profiles;

SELECT COUNT(*) AS total_viewership_records 
FROM workspace.default.bright_tv_viewership;


-- --------------------------------------------
-- DISTINCT VALUE ANALYSIS
-- --------------------------------------------
-- Identify unique channels available on the platform (~21 channels)
SELECT DISTINCT(Channel2) 
FROM workspace.default.bright_tv_viewership;

-- Identify unique subscribers (user base ≈ 1,000)
SELECT DISTINCT(UserID0) 
FROM workspace.default.bright_tv_viewership;


-- --------------------------------------------
-- DEMOGRAPHIC DATA EXPLORATION
-- --------------------------------------------
-- Review categorical distributions for profiling

-- Race categories present in dataset
SELECT DISTINCT(race) 
FROM workspace.default.bright_tv_user_profiles;

-- Province distribution (expected: 9 provinces + 'None')
SELECT DISTINCT(province) 
FROM workspace.default.bright_tv_user_profiles;

-- Gender categories
SELECT DISTINCT(gender) 
FROM workspace.default.bright_tv_user_profiles;


-- --------------------------------------------
-- AGE SEGMENTATION
-- --------------------------------------------
-- Create age bands using CASE logic
-- Supports demographic segmentation in downstream analysis
SELECT 
    Age,
    CASE
        WHEN Age = 0 THEN 'Infant (0)'
        WHEN Age BETWEEN 1 AND 12 THEN 'Kids (1–12)'
        WHEN Age BETWEEN 13 AND 20 THEN 'Teenagers (13–20)'
        WHEN Age BETWEEN 21 AND 44 THEN 'Young Adults (21–44)'
        WHEN Age BETWEEN 45 AND 64 THEN 'Mature Adults (45–64)'
        ELSE 'Senior (65+)'
    END AS age_category
FROM workspace.default.bright_tv_user_profiles;


-- --------------------------------------------
-- DATA CLEANING (NULL & INVALID VALUES)
-- --------------------------------------------
-- Confirm presence of nulls or placeholder values
SELECT DISTINCT(Gender) 
FROM workspace.default.bright_tv_user_profiles;

-- Standardise categorical fields:
-- Replace 'None' or empty values with 'Unknown'
SELECT
    COALESCE(NULLIF(gender, 'None'), 'Unknown') AS cleaned_gender,
    COALESCE(NULLIF(race, 'None'), 'Unknown') AS cleaned_race,
    COALESCE(NULLIF(province, 'None'), 'Unknown') AS cleaned_province
FROM workspace.default.bright_tv_user_profiles;


-- --------------------------------------------
-- DATA COVERAGE VALIDATION
-- --------------------------------------------
-- Confirm dataset spans Jan–Mar 2026 (3 months)
SELECT 
    MIN(RecordDate2) AS start_date,
    MAX(RecordDate2) AS end_date
FROM workspace.default.bright_tv_viewership;


-- ============================================
-- FINAL TRANSFORMATION QUERY (ANALYTICS READY)
-- ============================================

SELECT

-- --------------------------------------------
-- TIME STANDARDISATION
-- --------------------------------------------
-- Convert UTC timestamps to South African Standard Time (SAST)
from_utc_timestamp(RecordDate2, 'Africa/Johannesburg') AS sast_timestamp,

-- Extract date and time components for flexible analysis
to_date(from_utc_timestamp(RecordDate2, 'Africa/Johannesburg')) AS record_day,
date_format(from_utc_timestamp(RecordDate2, 'Africa/Johannesburg'), 'HH:mm:ss') AS record_time,


-- --------------------------------------------
-- DURATION TRANSFORMATION
-- --------------------------------------------
-- Convert duration into total minutes for aggregation
`Duration 2`,
ROUND(
    HOUR(`Duration 2`) * 60 
    + MINUTE(`Duration 2`) 
    + SECOND(`Duration 2`) / 60.0, 
2) AS duration_minutes,


-- --------------------------------------------
-- TEMPORAL FEATURES
-- --------------------------------------------
-- Enable trend analysis by day, month, and year
DAYNAME(record_day) AS day_name,
MONTHNAME(record_day) AS month_name,
YEAR(record_day) AS year_,
DAY(record_day) AS day_of_month_,


-- --------------------------------------------
-- CORE DIMENSIONS
-- --------------------------------------------
v.channel2,
p.age,


-- --------------------------------------------
-- DATA CLEANING (STANDARDISED DIMENSIONS)
-- --------------------------------------------
-- Remove blanks and 'None' values for consistency
COALESCE(NULLIF(NULLIF(TRIM(gender), 'None'), ''), 'Unknown') AS cleaned_gender,
COALESCE(NULLIF(NULLIF(TRIM(race), 'None'), ''), 'Unknown') AS cleaned_race,
COALESCE(NULLIF(NULLIF(TRIM(province), 'None'), ''), 'Unknown') AS cleaned_province,


-- --------------------------------------------
-- AGE SEGMENTATION (FINAL)
-- --------------------------------------------
CASE 
    WHEN Age = 0 THEN 'Infant (0)'
    WHEN Age BETWEEN 1 AND 12 THEN 'Kids (1–12)'
    WHEN Age BETWEEN 13 AND 19 THEN 'Teenagers (13–19)'
    WHEN Age BETWEEN 20 AND 35 THEN 'Young Adults (20–35)'
    WHEN Age BETWEEN 36 AND 64 THEN 'Mature Adults (36–64)'
    ELSE 'Senior (65+)'
END AS age_category,


-- --------------------------------------------
-- WEEKDAY VS WEEKEND CLASSIFICATION
-- --------------------------------------------
CASE
    WHEN DAYNAME(record_day) IN ('Sat', 'Sun') THEN 'Weekend'
    ELSE 'Weekday'
END AS day_classification,


-- --------------------------------------------
-- TIME-OF-DAY SEGMENTATION
-- --------------------------------------------
-- Create viewing buckets for behavioural analysis
CASE
    WHEN HOUR(from_utc_timestamp(RecordDate2, 'Africa/Johannesburg')) BETWEEN 6 AND 11 THEN 'Morning'
    WHEN HOUR(from_utc_timestamp(RecordDate2, 'Africa/Johannesburg')) BETWEEN 12 AND 16 THEN 'Afternoon'
    WHEN HOUR(from_utc_timestamp(RecordDate2, 'Africa/Johannesburg')) BETWEEN 17 AND 21 THEN 'Evening (Prime)'
    ELSE 'Night / Early Morning'
END AS time_bucket


-- --------------------------------------------
-- DATA INTEGRATION
-- --------------------------------------------
FROM workspace.default.bright_tv_viewership AS v
LEFT JOIN workspace.default.bright_tv_user_profiles AS p
ON v.UserID0 = p.UserID;